In [ ]:
%pip install certifi
%pip install opencv-python
%pip install pandas
%pip install httpx
%pip install scikit-learn

In [ ]:
import base64
import certifi
import json
import os
import sys
import cv2
import pandas as pd
import httpx  # using httpx instead of google.generativeai
import subprocess
import re
import shutil
import xml.etree.ElementTree as ET
import glob
import xml.etree.ElementTree as ET
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_NAME = "gemini-2.0-flash"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent"

def get_video_folders():
    valid_paths = []
    
    try:
        inputs_path = os.path.join(get_current_folder(), "inputs")
        if os.path.exists(inputs_path):
            with open(inputs_path, "r") as file:
                content = file.read().strip()
                if content:
                    # Parse all VIDEO_FOLDER_PATH variables from inputs file
                    for line in content.split('\n'):
                        if line.strip().startswith('VIDEO_FOLDER_PATH='):
                            path = line.split('=', 1)[1].strip().strip('"\'')
                            if path:
                                # Expand user home directory (~) and environment variables
                                expanded_path = os.path.expanduser(os.path.expandvars(path))
                                # Convert to absolute path and normalize
                                abs_path = os.path.abspath(expanded_path)
                                if os.path.exists(abs_path):
                                    if abs_path not in valid_paths:  # Avoid duplicates
                                        valid_paths.append(abs_path)
                                else:
                                    print(f"⚠️ Warning: Video folder path does not exist: {abs_path}")
    except Exception as e:
        print(f"⚠️ Warning: Could not read inputs file: {e}")
    
    # If we found valid paths from inputs file, return them
    if valid_paths:
        return valid_paths
    
    # If no valid paths found, return empty list
    print("❌ No valid video folder paths found in inputs file.")
    return []

def scan_for_videos(folder_paths):
    """Scan for video files in the given folder paths (top-level only, no subfolders)."""
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv', '.webm', '.m4v', '.3gp', '.mpg', '.mpeg'}
    all_video_paths = []
    
    for folder_path in folder_paths:
        try:
            print(f"🔍 Scanning folder: {folder_path} (top-level only)")
            video_count = 0
            
            # Scan top level files from provided folder.
            if os.path.exists(folder_path):
                for file in os.listdir(folder_path):
                    file_path = os.path.join(folder_path, file)
                    if os.path.isfile(file_path):
                        file_ext = os.path.splitext(file)[1].lower()
                        if file_ext in video_extensions:
                            video_path = os.path.abspath(file_path)
                            all_video_paths.append(video_path)
            else:
                print(f"⚠️ Folder does not exist: {folder_path}")
            
            print(f"✅ Found {video_count} video files in {folder_path}")
            
        except Exception as e:
            print(f"❌ Error scanning folder {folder_path}: {e}")
    
    print(f"🎬 Total unique videos found: {len(all_video_paths)}")
    return all_video_paths

def get_current_folder():
    # In Jupyter notebooks, use the current working directory
    return os.getcwd()

def get_tool_path(tool_name):
    if sys.platform == "win32" and not tool_name.endswith(".exe"):
        tool_name += ".exe"
    # In Jupyter notebooks, use the current working directory
    base_path = getattr(sys, '_MEIPASS', os.getcwd())
    return os.path.join(base_path, "tools", tool_name)

ffmpeg_path = get_tool_path("ffmpeg")
ffprobe_path = get_tool_path("ffprobe")

def get_settings_path():
    base_path = get_current_folder()
    return os.path.join(base_path, "settings.json")

def load_api_key():
    google_api_key = None
    settings_path = get_settings_path()

    if os.path.exists(settings_path):
        try:
            with open(settings_path, "r") as file:
                config = json.load(file)
                google_api_key = config.get("api_key", "").strip()
                if google_api_key:
                    print("✅ Google Gemini API Key Loaded")
                else:
                    print("❌ Google Gemini API Key is missing.")
        except Exception as e:
            print(f"❌ ERROR loading settings: {e}")
    else:
        print("❌ ERROR: settings.json file not found.")
    
    return google_api_key

def load_settings():
    """Load all settings from settings.json file."""
    settings = {
        "api_key": "",
        "apply_metadata": False,
        "export_format": "csv",
        "custom_prompt": "",
        "important_keywords": "",
        "creator": "",
        "copyright": ""
    }
    
    settings_path = get_settings_path()
    if os.path.exists(settings_path):
        try:
            with open(settings_path, "r") as file:
                config = json.load(file)
                settings.update(config)
                print("✅ Settings loaded successfully")
        except Exception as e:
            print(f"❌ ERROR loading settings: {e}")
    else:
        print("❌ ERROR: settings.json file not found.")
    
    return settings

def get_keyframes_folder():
    # Use local keyframes folder in the project directory
    folder = os.path.join(get_current_folder(), "keyframes")
    os.makedirs(folder, exist_ok=True)
    return folder

class KeyFrame:
    def __init__(self, keyframe_paths=[], duration=0, fps=0, resolution=""):
        self.keyframe_paths = keyframe_paths
        self.duration = duration
        self.fps = fps
        self.resolution = resolution

def extract_keyframes(video_path):
    """Extracts one keyframe at the midpoint of the video."""
    output_dir = get_keyframes_folder()
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps else 0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frame_index = total_frames // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ret, frame = cap.read()

    keyframe_paths = []
    if ret:
        keyframe_path = os.path.join(output_dir, f"{os.path.basename(video_path)}_keyframe_mid.jpg")
        cv2.imwrite(keyframe_path, frame)
        keyframe_paths.append(keyframe_path)

    cap.release()
    resolution = f"{width}x{height}"
    KeyFrameInstance = KeyFrame(keyframe_paths, duration, fps, resolution)
    
    return KeyFrameInstance

def describe_keyframes_with_gemini_flash(keyframe_paths, custom_prompt=None, output_display=None):
    api_key = load_api_key()
    headers = {
        "Content-Type": "application/json",
        "x-goog-api-key": api_key
    }
    descriptions = []
    prompt = custom_prompt or '''
    Analyze the image and deliver a strong description of what you see. From that description generate also a filename and tags.

    For filenames use identified names or descriptors.
    Scenary = The type of screen you identified like "Combat", "Party Setup", "Idle" (e.g. Character posing in front of a landscape), "Weapons", "Artifacts", "Constellations", "Talents", etc.
    Subjects = Can be Character names or item names.
    Apply this format to the filename and replace the placeholders with what you identified. FORMAT: [Scenary]_[Subject1]_[Subject2]_[Subject3]... e.g. PartySetup_Alhaitham_Baizhu_RaidenShogun_Xingqiu
    Do not use spaces or special characters, use underscores instead.

    For the tags provide a comma-separated list.
    '''

    for keyframe_path in keyframe_paths:
        msg = f"🔹 Sending {keyframe_path} to Gemini API"
        if output_display:
            output_display.append(msg)
        else:
            print(msg)

        with open(keyframe_path, "rb") as image_file:
            image_bytes = image_file.read()

        image_base64 = base64.b64encode(image_bytes).decode("utf-8")

        payload = {
            "contents": [
                {"parts": [
                    {"text": prompt},
                    {"inlineData": {"mimeType": "image/jpeg", "data": image_base64}}
                ]}
            ]
        }

        response = httpx.post(API_URL, headers=headers, json=payload, verify=certifi.where())
        if response.status_code == 200:
            result = response.json()
            description = result["candidates"][0]["content"]["parts"][0]["text"].strip()
            descriptions.append(description)
            if output_display:
                output_display.append(f"✅ AI Description: {description}")
            else:
                print(f"✅ AI Description: {description}")
        else:
            descriptions.append("Error")
            msg = f"❌ API Error {response.status_code}: {response.text}"
            if output_display:
                output_display.append(msg)
            else:
                print(msg)

    return descriptions

# Generate a master description
def generate_master_description(descriptions):
    if not descriptions:
        return "No description available"
    
    match = re.search(r"\*\*Description:\*\*\s*(.*?)\s*\*\*Filename", descriptions[0], re.DOTALL)
    if match:
        return match.group(1).strip()
    
    # Fallback: return the first valid description if regex doesn't match
    return descriptions

def extract_keywords(descriptions):
    if not descriptions:
        return "No description available"

    match = re.search(r"\*\*Tags:\*\*\s*(.*)", descriptions[0], re.DOTALL)
    if match:
        return match.group(1).strip()

# Extract keywords for filename generation
def extract_filename(descriptions):
    if not descriptions:
        return "unknown_video"
    
    match = re.search(r"\*\*Filename:\*\*\s*(.*?)\s*\*\*Tags", descriptions[0], re.DOTALL)
    if match:
        filename = match.group(1).strip()
        return filename
    
    print(descriptions[0])
    
    # Fallback: generate filename from first few words of descriptions
    words = " ".join(descriptions).split()[:3]
    clean_words = [re.sub(r'[^\w\s-]', '', word) for word in words]
    return "_".join(clean_words).lower() or "unknown_video"

def process_videos(video_paths, selected_options, output_display=None):
    api_key = load_api_key()
    if not api_key:
        msg = "❌ ERROR: No API key provided! AI processing will fail."
        if output_display:
            output_display.append(msg)
        else:
            print(msg)
        return {}

    custom_prompt = selected_options.get("custom_prompt", "")
    important_keywords = selected_options.get("important_keywords", "")
    creator = selected_options.get("creator", "")
    copyright_text = selected_options.get("copyright", "")

    video_metadata = {}
    
    print(f"🐛 DEBUG: About to process {len(video_paths)} videos:")
    for i, path in enumerate(video_paths):
        print(f"  {i+1}: {path}")
    
    for i, video_path in enumerate(video_paths):
        msg = f"🔹 Processing video: {video_path}"

        print(msg)

        data = extract_keyframes(video_path)
        keyframe_paths = data.keyframe_paths
        duration = data.duration
        fps = data.fps
        resolution = data.resolution

        print("length of keyframes:", len(keyframe_paths))
        descriptions = describe_keyframes_with_gemini_flash(keyframe_paths, custom_prompt=custom_prompt, output_display=output_display)
        master_description = generate_master_description(descriptions)

        # Check if description generation failed
        if not master_description:
            msg = f"❌ AI failed to generate a valid description for {video_path}"
            print(msg)
            
        final_keywords = extract_keywords(descriptions)
        shortened_filename = f"{i}_{extract_filename(descriptions)}.mp4"

        video_metadata[video_path] = {
            "master_description": master_description,
            "shortened_filename": shortened_filename,
            "duration": duration,
            "fps": fps,
            "resolution": resolution,
            "keywords": final_keywords,
            "creator": creator,
            "copyright": copyright_text
        }

    if not video_metadata:
        msg = "❌ No videos were successfully processed. Please check for API errors or keyframe extraction issues."
        if output_display:
            output_display.append(msg)
        else:
            print(msg)

    msg = f"🔹 Final metadata before saving: {video_metadata}"
    if output_display:
        output_display.append(msg)
    else:
        print(msg)

    save_metadata_to_csv(video_metadata)

    return video_metadata

def get_metadata_path():
    """Returns the absolute path for metadata.csv."""
    base_path = get_current_folder()
    os.makedirs(base_path, exist_ok=True)  # Ensure the folder exists
    return os.path.join(base_path, "metadata.csv")

def ensure_metadata_file():
    """Creates metadata.csv if it does not exist."""
    metadata_path = get_metadata_path()
    if not os.path.exists(metadata_path):
        print("⚠️ metadata.csv not found. Creating a new empty file.")
        df = pd.DataFrame(columns=["video_filename", "duration", "fps", "resolution", "description", "shortened_filename"])
        df.to_csv(metadata_path, index=False)
    return metadata_path

# ✅ Call this function at startup
def save_metadata_to_csv(video_metadata):
    metadata_path = get_metadata_path()

    print(f"🔹 Saving metadata to: {metadata_path}")
    print(f"🔹 Video metadata received: {video_metadata}")

    if not video_metadata:
        print("❌ No metadata received, skipping save.")
        return

    try:
        data = [
            {
                "video_filename": video_path,
                "duration": metadata["duration"],
                "fps": metadata["fps"],
                "resolution": metadata["resolution"],
                "description": metadata["master_description"],
                "shortened_filename": metadata["shortened_filename"]
            }
            for video_path, metadata in video_metadata.items()
        ]

        if not data:
            print("⚠️ No valid data found, skipping write operation.")
            return

        df = pd.DataFrame(data)
        df.to_csv(metadata_path, index=False)
        print("✅ Metadata saved successfully!")

    except Exception as e:
        print(f"❌ ERROR saving metadata: {e}")

def load_video_metadata():
    """Reads metadata.csv and structures it for exporting."""
    metadata_path = get_metadata_path()
    
    if not os.path.exists(metadata_path):
        return {}

    metadata_df = pd.read_csv(metadata_path)
    video_metadata = {}

    for _, row in metadata_df.iterrows():
        video_metadata[row["video_filename"]] = {
            "shortened_filename": row["shortened_filename"],
            "master_description": row["description"],
            "duration": row["duration"],
            "fps": row["fps"],
            "resolution": row["resolution"]
        }
    
    return video_metadata

def apply_metadata_to_videos(csv_file="metadata.csv", output_folder="output"):
    os.makedirs(output_folder, exist_ok=True)  # Ensure the folder exists

    metadata_df = pd.read_csv(csv_file)

    for index, row in metadata_df.iterrows():
        original_video_file = row['video_filename']
        shortened_filename = row['shortened_filename']
        description = row['description']
        
        if shortened_filename == "error":
            print(f"⚠️ Error detected for video: {original_video_file}. Skipping.")
            continue

        # Define new output path using the selected folder
        output_video_path = os.path.join(output_folder, shortened_filename)

        # Ensure file exists before processing
        if not os.path.exists(original_video_file):
            print(f"❌ File not found: {original_video_file}. Skipping.")
            continue

        # ✅ FFmpeg Rewrap (Preserves metadata, enables fast start)
        ffmpeg_command = [
            ffmpeg_path,
            "-hide_banner", "-loglevel", "error",   # Reduce output noise
            "-i", original_video_file,
            "-movflags", "+faststart",
            "-c", "copy",
            output_video_path
        ]

        print(f"▶️ Running FFmpeg:\n{' '.join(ffmpeg_command)}")

        result = subprocess.run(ffmpeg_command, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"❌ FFmpeg Error:\n{result.stderr}")
        else:
            print(f"✅ Video rewrapped and saved: {output_video_path}")

        # Apply metadata using ExifTool
        #exiftool_path = shutil.which("exiftool") or "/opt/homebrew/bin/exiftool"  # Adjust based on your system

        #exiftool_command = [
            #exiftool_path,
            #f"-QuickTime:Title={shortened_filename}",
            #"-QuickTime:Artist=AI Video Tagger",
            #f"-QuickTime:Description={description}",
            #"-overwrite_original",
            #output_video_path
        #]
        #subprocess.run(exiftool_command, capture_output=True, text=True, check=False)

        print(f"✅ Metadata applied to {output_video_path}")

# Export CSV for editors
def export_csv(video_metadata, filename, columns, delimiter=","):
    if not video_metadata:
        video_metadata = load_video_metadata()

    data = [
        {col: metadata.get(col, "") for col in columns}
        for metadata in video_metadata.values()
    ]
    pd.DataFrame(data).to_csv(filename, index=False, sep=delimiter)
    print(f"✅ Exported: {filename}")

def export_csv_davinci(video_metadata):
    """Exports metadata in CSV format for DaVinci Resolve."""
    filename = os.path.join(get_current_folder(), "metadata_davinci.csv")

    data = []
    for video_path, metadata in video_metadata.items():
        row = {
            "File Name": metadata.get("shortened_filename", os.path.basename(video_path)),
            "Description": metadata.get("master_description", ""),
            "Keywords": metadata.get("keywords", ""),
            "Scene": "",  # DaVinci allows empty Scene
            "Shot": "",   # DaVinci allows empty Shot
            "Notes": ""
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print(f"✅ DaVinci Resolve CSV exported: {filename}")

def export_csv_premiere(video_metadata):
    """Exports metadata in CSV format for Adobe Premiere Pro."""
    filename = os.path.join(get_current_folder(), "metadata_premiere.csv")

    data = []
    for video_path, metadata in video_metadata.items():
        row = {
            "Clip Name": metadata.get("shortened_filename", os.path.basename(video_path)),
            "Description": metadata.get("master_description", ""),
            "Tags": metadata.get("keywords", ""),
            "Scene": "",
            "Shot": "",
            "Notes": ""
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print(f"✅ Premiere Pro CSV exported: {filename}")

def export_csv_avid(video_metadata):
    """Exports metadata in TAB-delimited format for Avid Media Composer."""
    filename = os.path.join(get_current_folder(), "metadata_avid.txt")

    data = []
    for video_path, metadata in video_metadata.items():
        row = [
            metadata.get("shortened_filename", os.path.basename(video_path)),
            metadata.get("master_description", ""),
            metadata.get("keywords", ""),
            "",  # Scene
            "",  # Shot
            ""  #Notes
        ]
        data.append(row)

    df = pd.DataFrame(data, columns=["File Name", "Description", "Keywords", "Scene", "Shot", "Notes"])
    df.to_csv(filename, index=False, sep="\t")  # TAB-separated
    print(f"✅ Avid Media Composer CSV exported: {filename}")

def export_xml_fcp(video_metadata):
    """Exports metadata in XML format for Final Cut Pro."""
    filename = os.path.join(get_current_folder(), "metadata_fcpx.xml")

    root = ET.Element("fcpxml", version="1.10")
    resources = ET.SubElement(root, "resources")

    for video_path, metadata in video_metadata.items():
        clip = ET.SubElement(root, "clip", name=metadata.get("shortened_filename", os.path.basename(video_path)))
        ET.SubElement(clip, "description").text = metadata.get("master_description", "")
        ET.SubElement(clip, "keywords").text = metadata.get("keywords", ""),
        ET.SubElement(clip, "notes").text = "AI-generated metadata"

    tree = ET.ElementTree(root)
    tree.write(filename, encoding="utf-8", xml_declaration=True)
    print(f"✅ Final Cut Pro XML exported: {filename}")

def export_csv_iptc(video_metadata, parent=None):
    """Export metadata to a CSV using the IPTC standard fields."""
    if not video_metadata:
        video_metadata = load_video_metadata()

    filename = os.path.join(get_current_folder(), "iptc_metadata.csv")

    # Standard IPTC fields we'll use
    columns = {
        "Object Name": "shortened_filename",
        "Caption/Abstract": "master_description",
        "Keywords": "keywords",
        "Writer/Editor": "creator",
        "Copyright Notice": "copyright",
        "Source": "VideoTaggerApp",
        "Headline": "shortened_filename"
    }

    data = []

    for path, metadata in video_metadata.items():
        row = {}
        for iptc_field, meta_key in columns.items():
            if meta_key in metadata:
                row[iptc_field] = metadata[meta_key]
            else:
                row[iptc_field] = ""
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print(f"✅ IPTC CSV exported to: {filename}")

def clear_keyframes_folder():
    folder = get_keyframes_folder()
    for f in glob.glob(os.path.join(folder, "*.jpg")):
        os.remove(f)

def export_metadata_csv(video_metadata, export_format):
    """Export metadata using the specified format."""
    if not video_metadata:
        video_metadata = load_video_metadata()
    
    if not video_metadata:
        print("❌ No metadata available for export.")
        return
    
    export_format = export_format.lower()
    
    if export_format == "davinci":
        export_csv_davinci(video_metadata)
    elif export_format == "premiere":
        export_csv_premiere(video_metadata)
    elif export_format == "avid":
        export_csv_avid(video_metadata)
    elif export_format == "fcpx" or export_format == "finalcut":
        export_xml_fcp(video_metadata)
    elif export_format == "iptc":
        export_csv_iptc(video_metadata)
    elif export_format == "csv":
        # Use generic CSV export
        columns = ["shortened_filename", "master_description", "keywords", "duration", "fps", "resolution"]
        filename = os.path.join(get_current_folder(), "metadata_export.csv")
        export_csv(video_metadata, filename, columns)
    else:
        print(f"❌ Unknown export format: {export_format}. Available formats: davinci, premiere, avid, fcpx, iptc, csv")
        print("Using default CSV format.")
        columns = ["shortened_filename", "master_description", "keywords", "duration", "fps", "resolution"]
        filename = os.path.join(get_current_folder(), "metadata_export.csv")
        export_csv(video_metadata, filename, columns)

def main():
    """Main entry point for video processing workflow."""
    print("🎬 Starting Video Metadata Processing...")
    
    # Pre-flight checks
    print("\n🔍 Running pre-flight checks...")
    
    # Load settings
    settings = load_settings()
    
    # Validate API key
    if not settings.get("api_key"):
        print("❌ ERROR: No API key found in settings.json. Cannot proceed with AI processing.")
        return
    
    # Get video folders from inputs file
    print("\n📁 Getting video folders...")
    video_folders = get_video_folders()
    
    if not video_folders:
        print("❌ No valid video folders found. Please check your inputs file.")
        return
    
    print(f"✅ Found {len(video_folders)} valid folder(s)")
    
    # Scan for video files
    print("\n🔍 Scanning for video files...")
    video_paths = scan_for_videos(video_folders)
    
    if not video_paths:
        print("❌ No video files found in the specified folders.")
        return
    
    print(f"✅ Found {len(video_paths)} video file(s) to process")
    
    # Process videos with AI
    print(f"\n🤖 Processing {len(video_paths)} videos with AI...")
    selected_options = {
        "custom_prompt": settings.get("custom_prompt", ""),
        "important_keywords": settings.get("important_keywords", ""),
        "creator": settings.get("creator", ""),
        "copyright": settings.get("copyright", "")
    }
    
    video_metadata = process_videos(video_paths, selected_options)
    
    if not video_metadata:
        print("❌ No videos were successfully processed.")
        return
    
    success_count = len(video_metadata)
    total_count = len(video_paths)
    print(f"✅ Successfully processed {success_count}/{total_count} videos")
    
    # Apply metadata to videos (optional)
    if settings.get("apply_metadata", False):
        print("\n📝 Applying metadata to video files...")
        apply_metadata_to_videos()
    else:
        print("\n⏭️ Skipping metadata application (disabled in settings)")
    
    # Export metadata
    print("\n📤 Exporting metadata...")
    export_format = settings.get("export_format", "csv")
    export_metadata_csv(video_metadata, export_format)
    
    # Cleanup keyframes
    print("\n🧹 Cleaning up temporary keyframes...")
    try:
        clear_keyframes_folder()
        print("✅ Keyframes cleaned up successfully")
    except Exception as e:
        print(f"⚠️ Warning: Could not clean up keyframes: {e}")
    
    print(f"\n🎉 Video processing workflow completed!")
    print(f"📊 Final results: {success_count}/{total_count} videos processed successfully")

# Only run main() if this script is executed directly (not imported)
if __name__ == "__main__":
    main()
